# Multimodal Explainable AI for Early Diabetes Prediction
## Full Pipeline — Phases 1–5

| Phase | Description |
|---|---|
| 1 | Data loading, preprocessing, feature engineering |
| 2 | Synthetic clinical notes + ClinicalBERT embeddings |
| 3 | Baseline models + Early/Late Fusion + evaluation |
| 4 | SHAP explainability + LLM recommendations |
| 5 | Publication-ready figures and tables |

> **Dataset:** Diabetes 130-US Hospitals 1999–2008 (UCI, ID=296)  
> **Reproducibility:** `RANDOM_SEED = 42` throughout.

## 1. Imports & global config

All libraries for the entire pipeline imported here once.

In [2]:
import os, json, time, warnings, pickle, shutil, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm.auto import tqdm

from ucimlrepo import fetch_ucirepo
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    average_precision_score, roc_curve, precision_recall_curve,
    confusion_matrix, ConfusionMatrixDisplay,
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.calibration import calibration_curve
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from groq import Groq, RateLimitError

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModel
import shap

warnings.filterwarnings("ignore")
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

plt.rcParams.update({
    "figure.facecolor":"white","axes.facecolor":"#F8F9FB",
    "axes.grid":True,"grid.alpha":0.4,
    "axes.spines.top":False,"axes.spines.right":False,"font.size":11,
})
PALETTE = {
    "neg":"#4A90D9","pos":"#E05C5C","neu":"#7F77DD",
    "lr":"#4A90D9","rf":"#56B87A","xgb":"#E0AA00",
    "early":"#E05C5C","late":"#9B59B6","tab":"#7F77DD",
}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs("data", exist_ok=True)
os.makedirs("data/models", exist_ok=True)
os.makedirs("data/figures", exist_ok=True)
os.makedirs("data/paper", exist_ok=True)
print(f"Ready. RANDOM_SEED={RANDOM_SEED} | device={device}")

KeyboardInterrupt: 

## 2. Google Drive — mount & restore

Run at the start of **every new Colab session** to restore saved artefacts.

In [4]:
from google.colab import drive, userdata
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/diabetes_project/data"
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/models", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/paper", exist_ok=True)

RESTORE_FILES = [
    "scaler.pkl","feature_names.json","diabetes_clean.parquet",
    "X_train.npy","X_val.npy","X_test.npy",
    "y_train.npy","y_val.npy","y_test.npy",
    "text_embeddings.npy","synthetic_notes_final.csv",
    "synthetic_notes.csv","notes_checkpoint.csv",
    "results_summary.csv","shap_values_xgb.npy",
    "recommendations_sample.csv",
]
for fname in RESTORE_FILES:
    src, dst = f"{DRIVE_DIR}/{fname}", f"data/{fname}"
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy2(src, dst); print(f"  Restored: {fname}")

for mname in ["xgboost.json","logistic_regression.pkl",
              "random_forest.pkl","early_fusion.pt","late_fusion.pt"]:
    src, dst = f"{DRIVE_DIR}/models/{mname}", f"data/models/{mname}"
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy2(src, dst); print(f"Restored model: {mname}")

print("Restore complete.")

Mounted at /content/drive


FileNotFoundError: [Errno 2] No such file or directory: 'data/scaler.pkl'

## Publication plot style

In [ ]:
PAPER_DIR = "data/paper"
FIG_DIR   = "data/figures"
os.makedirs(PAPER_DIR, exist_ok=True)
os.makedirs(FIG_DIR,   exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/paper", exist_ok=True)

plt.rcParams.update({
    "figure.facecolor":"white","axes.facecolor":"#F8F9FB",
    "axes.grid":True,"grid.alpha":0.35,
    "axes.spines.top":False,"axes.spines.right":False,
    "font.family":"DejaVu Sans","font.size":11,
    "axes.titlesize":12,"axes.labelsize":11,"legend.fontsize":9,
    "savefig.dpi":200,"savefig.bbox":"tight","savefig.facecolor":"white",
})
print("Publication style configured.")

## 39. Figure 1 — Master model comparison (4 panels)

In [ ]:
ef_auc = roc_auc_score(y_test_mm, ef_test_prob)
lf_auc = roc_auc_score(y_test_mm, lf_test_prob)
tab_only_auc = all_results_df.loc["Tabular-Only NN","auc_roc"] if "Tabular-Only NN" in all_results_df.index else None

model_cfgs=[
    ("Logistic Regression",lr_model.predict_proba(X_test)[:,1], y_test,PALETTE["lr"],"--",1.5),
    ("Random Forest", rf_model.predict_proba(X_test)[:,1], y_test,PALETTE["rf"],"--",1.5),
    ("XGBoost", xgb_model.predict_proba(X_test)[:,1],y_test, PALETTE["xgb"],"--",1.5),
    ("Early Fusion NN",ef_test_prob, y_test_mm,PALETTE["early"],"-",2.5),
    ("Late Fusion NN", lf_test_prob, y_test_mm,PALETTE["late"],"-",2.5),
]

fig=plt.figure(figsize=(18,14))
gs=gridspec.GridSpec(2,2,hspace=0.38,wspace=0.32)
ax_bar=fig.add_subplot(gs[0,0]); ax_roc=fig.add_subplot(gs[0,1])
ax_pr=fig.add_subplot(gs[1,0]);  ax_abl=fig.add_subplot(gs[1,1])

model_names=[c[0] for c in model_cfgs]
model_aucs=[roc_auc_score(c[2],c[1]) for c in model_cfgs]
model_colors=[c[3] for c in model_cfgs]
bars=ax_bar.barh(model_names,model_aucs,color=model_colors,edgecolor="white",linewidth=1.5)
ax_bar.set_xlim(0.45,1.0); ax_bar.set_xlabel("AUC-ROC")
ax_bar.set_title("(A) AUC-ROC — All Models",fontweight="bold")
ax_bar.axvline(0.5,color="gray",ls=":",lw=1)
best_i=int(np.argmax(model_aucs))
for i,(bar,val) in enumerate(zip(bars,model_aucs)):
    ax_bar.text(val+0.003,bar.get_y()+bar.get_height()/2,f"{val:.4f}",
                va="center",fontsize=9,fontweight="bold" if i==best_i else "normal")

for nm,prob,yt,clr,ls,lw in model_cfgs:
    fpr,tpr,_=roc_curve(yt,prob); auc=roc_auc_score(yt,prob)
    ax_roc.plot(fpr,tpr,color=clr,lw=lw,ls=ls,label=f"{nm} ({auc:.3f})")
ax_roc.plot([0,1],[0,1],"k:",lw=1)
ax_roc.set_xlabel("FPR"); ax_roc.set_ylabel("TPR")
ax_roc.set_title("(B) ROC Curves",fontweight="bold"); ax_roc.legend(loc="lower right",fontsize=8)

for nm,prob,yt,clr,ls,lw in model_cfgs:
    prec,rec,_=precision_recall_curve(yt,prob); ap=average_precision_score(yt,prob)
    ax_pr.plot(rec,prec,color=clr,lw=lw,ls=ls,label=f"{nm} ({ap:.3f})")
ax_pr.axhline(y_test.mean(),color="gray",ls=":",lw=1,label=f"No-skill ({y_test.mean():.2f})")
ax_pr.set_xlabel("Recall"); ax_pr.set_ylabel("Precision")
ax_pr.set_title("(C) Precision-Recall Curves",fontweight="bold"); ax_pr.legend(fontsize=8)

abl_names=["XGBoost\n(tabular)","Tabular-Only\nNN","Early Fusion\nNN","Late Fusion\nNN"]
abl_aucs=[roc_auc_score(y_test,xgb_model.predict_proba(X_test)[:,1]),
          tab_only_auc if tab_only_auc else 0.0, ef_auc, lf_auc]
abl_colors=[PALETTE["xgb"],PALETTE["tab"],PALETTE["early"],PALETTE["late"]]
abl_bars=ax_abl.bar(abl_names,abl_aucs,color=abl_colors,edgecolor="white",linewidth=1.5)
ax_abl.set_ylim(min(v for v in abl_aucs if v>0)-0.05,max(abl_aucs)+0.05)
ax_abl.set_ylabel("AUC-ROC"); ax_abl.set_title("(D) Ablation: Text Modality",fontweight="bold")
if tab_only_auc:
    delta_abl=ef_auc-tab_only_auc
    ax_abl.annotate(f"Δ={delta_abl:+.4f}",xy=(2,ef_auc),xytext=(1.4,ef_auc+0.02),
                    arrowprops=dict(arrowstyle="->",color="black"),fontsize=9,fontweight="bold")
for bar,val in zip(abl_bars,abl_aucs):
    if val>0: ax_abl.text(bar.get_x()+bar.get_width()/2,val+0.003,
                          f"{val:.4f}",ha="center",fontsize=9,fontweight="bold")

fig.suptitle("Figure 1 — Model Performance Overview",fontsize=15,fontweight="bold",y=1.01)
plt.savefig(f"{PAPER_DIR}/figure1_model_comparison.png")
plt.show(); print("Saved: figure1_model_comparison.png")

## 40. Figure 2 — SHAP global explainability

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(18,7))
plt.sca(axes[0])
shap.summary_plot(xgb_shap_values,X_test_sample,feature_names=FEATURE_COLS,
                  max_display=15,show=False,plot_size=None)
axes[0].set_title("(A) SHAP Summary — Direction & Magnitude",fontweight="bold")
top15=importance_df.head(15)
axes[1].barh(top15["feature"][::-1],top15["mean_abs_shap"][::-1],color="#7F77DD",edgecolor="white",linewidth=1.2)
for i,(feat,val) in enumerate(zip(top15["feature"][::-1],top15["mean_abs_shap"][::-1])):
    axes[1].text(val+0.0003,i,f"{val:.4f}",va="center",fontsize=8)
axes[1].set_xlabel("Mean |SHAP Value|")
axes[1].set_title("(B) Global Feature Importance Ranking",fontweight="bold")
fig.suptitle("Figure 2 — SHAP Explainability (XGBoost)",fontsize=14,fontweight="bold")
plt.tight_layout()
plt.savefig(f"{PAPER_DIR}/figure2_shap_global.png")
plt.show(); print("Saved: figure2_shap_global.png")

## 41. Figure 3 — Patient explanation + LLM recommendation

In [ ]:
hr_rec=rec_df[rec_df["true_label"]==1]["recommendation"].iloc[0] if len(rec_df[rec_df["true_label"]==1])>0 else "N/A"
lr_rec=rec_df[rec_df["true_label"]==0]["recommendation"].iloc[0] if len(rec_df[rec_df["true_label"]==0])>0 else "N/A"

fig,axes=plt.subplots(1,2,figsize=(18,9)); fig.subplots_adjust(bottom=0.28)
for ax,pidx,label,rec_text,tc in [
    (axes[0],hr_idx,"HIGH RISK",hr_rec,PALETTE["pos"]),
    (axes[1],lr_idx,"LOW RISK", lr_rec,PALETTE["neg"]),
]:
    sv=xgb_shap_values[pidx]; fv=X_test_sample[pidx]
    risk=float(xgb_model.predict_proba(X_test_sample[pidx:pidx+1])[:,1][0])
    order=np.argsort(np.abs(sv))[::-1][:10]
    names=[FEATURE_COLS[i] for i in order]; vals=sv[order]; fvals=fv[order]
    colors=[PALETTE["pos"] if v>0 else PALETTE["neg"] for v in vals]
    ax.barh(range(10),vals[::-1],color=colors[::-1],edgecolor="white")
    ax.set_yticks(range(10))
    ax.set_yticklabels([f"{n}={fvals[9-i]:.2f}" for i,n in enumerate(names[::-1])],fontsize=8)
    ax.axvline(0,color="black",lw=0.8); ax.set_xlabel("SHAP Value")
    ax.set_title(f"[{label}]  Predicted risk: {risk:.1%}",fontweight="bold",color=tc,fontsize=11)
    wrapped="\n".join([rec_text[i:i+90] for i in range(0,len(rec_text),90)])
    ax.text(0.5,-0.30,f"Clinical recommendation:\n{wrapped}",transform=ax.transAxes,
            fontsize=8,ha="center",va="top",
            bbox=dict(boxstyle="round,pad=0.5",facecolor="#EEF2FF",edgecolor="#AABBDD"))

fig.suptitle("Figure 3 — Patient-Level SHAP + LLM Recommendation",fontsize=13,fontweight="bold")
plt.savefig(f"{PAPER_DIR}/figure3_patient_explanation.png")
plt.show(); print("Saved: figure3_patient_explanation.png")

## 42. Figure 4 — Fusion strategy analysis

In [ ]:
fig=plt.figure(figsize=(18,5)); gs=gridspec.GridSpec(1,3,wspace=0.35)
ax_met=fig.add_subplot(gs[0,0]); ax_dist=fig.add_subplot(gs[0,1]); ax_cal=fig.add_subplot(gs[0,2])

ef_row=all_results_df.loc["Early Fusion NN"] if "Early Fusion NN" in all_results_df.index else None
lf_row=all_results_df.loc["Late Fusion NN"]  if "Late Fusion NN"  in all_results_df.index else None
if ef_row is not None and lf_row is not None:
    fmet=["auc_roc","pr_auc","f1","precision","recall"]; flbl=["AUC-ROC","PR-AUC","F1","Precision","Recall"]
    x=np.arange(len(fmet)); w=0.35
    ax_met.bar(x-w/2,[ef_row[m] for m in fmet],width=w,label="Early Fusion",color=PALETTE["early"],edgecolor="white")
    ax_met.bar(x+w/2,[lf_row[m] for m in fmet],width=w,label="Late Fusion", color=PALETTE["late"], edgecolor="white")
    ax_met.set_xticks(x); ax_met.set_xticklabels(flbl,fontsize=9)
    ax_met.set_ylim(0,1.0); ax_met.set_ylabel("Score")
    ax_met.set_title("(A) Early vs Late — All Metrics",fontweight="bold"); ax_met.legend()

for prob,clr,lbl,ls in [(ef_test_prob,PALETTE["early"],"Early Fusion","-"),(lf_test_prob,PALETTE["late"],"Late Fusion","--")]:
    ax_dist.hist(prob[y_test_mm==0],bins=40,alpha=0.45,color=clr,histtype="step",linewidth=2,linestyle=ls,label=f"{lbl}|Low")
    ax_dist.hist(prob[y_test_mm==1],bins=40,alpha=0.35,color=clr,histtype="stepfilled",linewidth=1,linestyle=ls,label=f"{lbl}|High")
ax_dist.set_xlabel("Predicted Probability"); ax_dist.set_ylabel("Count")
ax_dist.set_title("(B) Probability Distribution",fontweight="bold"); ax_dist.legend(fontsize=7)

for prob,clr,nm in [(ef_test_prob,PALETTE["early"],"Early Fusion"),
                     (lf_test_prob,PALETTE["late"],"Late Fusion"),
                     (xgb_model.predict_proba(X_test_tab_mm)[:,1],PALETTE["xgb"],"XGBoost")]:
    try:
        fp,mp=calibration_curve(y_test_mm,prob,n_bins=8)
        ax_cal.plot(mp,fp,marker="o",color=clr,lw=2,markersize=5,label=nm)
    except: pass
ax_cal.plot([0,1],[0,1],"k:",lw=1,label="Perfect")
ax_cal.set_xlabel("Mean Predicted Probability"); ax_cal.set_ylabel("Fraction Positives")
ax_cal.set_title("(C) Calibration Curves",fontweight="bold"); ax_cal.legend(fontsize=8)

fig.suptitle("Figure 4 — Fusion Strategy Analysis",fontsize=13,fontweight="bold")
plt.savefig(f"{PAPER_DIR}/figure4_fusion_analysis.png")
plt.show(); print("Saved: figure4_fusion_analysis.png")

## 43. Figure 5 — Confusion matrices

> Uses optimal thresholds for neural networks.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,5))
for ax,nm,prob,yt,thresh in [
    (axes[0],"Early Fusion NN",ef_test_prob,y_test_mm,ef_threshold),
    (axes[1],"Late Fusion NN", lf_test_prob,y_test_mm,lf_threshold),
]:
    cm=confusion_matrix(yt,(prob>=thresh).astype(int))
    ConfusionMatrixDisplay(cm,display_labels=["Low Risk","High Risk"]).plot(ax=ax,colorbar=False,cmap="Blues")
    ax.set_title(f"{nm}\nAUC={roc_auc_score(yt,prob):.4f}  threshold={thresh}",fontweight="bold")
plt.suptitle("Figure 5 — Confusion Matrices (Test Set)",fontsize=12,fontweight="bold")
plt.tight_layout()
plt.savefig(f"{PAPER_DIR}/figure5_confusion_matrices.png")
plt.show(); print("Saved: figure5_confusion_matrices.png")

## 44. Figure 6 — SHAP dependence plots

In [ ]:
top3_idx=[FEATURE_COLS.index(f) for f in importance_df["feature"].head(3).tolist() if f in FEATURE_COLS]
if len(top3_idx)>=2:
    fig,axes=plt.subplots(1,len(top3_idx),figsize=(15,5))
    if len(top3_idx)==1: axes=[axes]
    for ax,fi in zip(axes,top3_idx):
        shap.dependence_plot(fi,xgb_shap_values,X_test_sample,feature_names=FEATURE_COLS,ax=ax,show=False)
        ax.set_title(f"Dependence: {FEATURE_COLS[fi]}",fontweight="bold",fontsize=10)
    plt.suptitle("Figure 6 — SHAP Dependence Plots (Top 3)",fontsize=12,fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"{PAPER_DIR}/figure6_shap_dependence.png")
    plt.show(); print("Saved: figure6_shap_dependence.png")

## 45. Table 1 — Final results

In [ ]:
print("\n"+"="*72)
print("TABLE 1 — Model Comparison (Test Set)")
print("="*72)
col_order=["auc_roc","pr_auc","f1","precision","recall"]
col_names=["AUC-ROC","PR-AUC","F1","Precision","Recall"]
table_df=all_results_df[col_order].copy(); table_df.columns=col_names
table_df=table_df.sort_values("AUC-ROC",ascending=False)
print(f"\n{'Model':<25}",end="")
for c in col_names: print(f"{c:>12}",end="")
print()
print("-"*85)
best_vals=table_df.max()
for model,row in table_df.iterrows():
    tag=" *" if model in ["Early Fusion NN","Late Fusion NN","Tabular-Only NN","XGBoost (aligned)"] else "  "
    print(f"{model:<25}",end="")
    for c in col_names:
        val=row[c]; star="*" if val==best_vals[c] else " "
        print(f"{star}{val:.4f}{'':<5}",end="")
    print(tag)
print("\n* Best in column  ** Multimodal test subset")
table_df.to_csv(f"{PAPER_DIR}/table1_results.csv")
print("Saved: table1_results.csv")

## 46. Table 2 — SHAP feature importance

In [ ]:
CLINICAL_NOTES={
    "num_medications": "Polypharmacy — surrogate for disease complexity",
    "time_in_hospital": "Longer stays indicate more severe episodes",
    "number_inpatient": "Prior hospitalizations predict future readmission",
    "number_diagnoses": "Comorbidity burden increases risk",
    "num_lab_procedures":"High lab count reflects acute management needs",
    "A1Cresult": "HbA1c elevation — direct glycemic control marker",
    "max_glu_serum": "Acute hyperglycemia during admission",
    "insulin": "Insulin dose change signals instability",
    "number_emergency": "Prior ED visits indicate fragile health status",
    "diabetesMed": "Active diabetes medication management",
}
print("\n"+"="*72)
print("TABLE 2 — SHAP Feature Importance")
print("="*72)
print(f"\n{'Rank':<6}{'Feature':<28}{'Mean |SHAP|':>12}  Clinical note")
print("-"*78)
for rank,(_,row) in enumerate(importance_df.head(15).iterrows(),1):
    note=CLINICAL_NOTES.get(row["feature"],"—")
    print(f"{rank:<6}{row['feature']:<28}{row['mean_abs_shap']:>12.4f}  {note}")
importance_df.head(15).to_csv(f"{PAPER_DIR}/table2_shap_importance.csv",index=False)
print("Saved: table2_shap_importance.csv")

## 47. Table 3 — Recommendation quality

In [ ]:
rec_df["word_count"]=rec_df["recommendation"].str.split().str.len()
KWS=["monitor","follow","refer","adjust","intervention","consult","review","reassess","specialist","discharge"]
rows=[]
for kw in KWS:
    hr=rec_df[rec_df["true_label"]==1]["recommendation"].str.lower().str.contains(kw).mean()
    lr=rec_df[rec_df["true_label"]==0]["recommendation"].str.lower().str.contains(kw).mean()
    if hr>0.05 or lr>0.05:
        rows.append({"keyword":kw,"high_risk_%":hr*100,"low_risk_%":lr*100,"delta_%":(hr-lr)*100})
kw_df=pd.DataFrame(rows).sort_values("delta_%",ascending=False)
print("\n"+"="*60); print("TABLE 3 — Recommendation Quality"); print("="*60)
print(f"\n{'Keyword':<20}{'High Risk %':>12}{'Low Risk %':>12}{'Delta %':>10}"); print("-"*56)
for _,r in kw_df.iterrows():
    print(f"{r['keyword']:<20}{r['high_risk_%']:>11.1f}%{r['low_risk_%']:>11.1f}%{r['delta_%']:>+9.1f}%")
print(f"\nMean words — High: {rec_df[rec_df['true_label']==1]['word_count'].mean():.1f}")
print(f"Mean words — Low : {rec_df[rec_df['true_label']==0]['word_count'].mean():.1f}")
kw_df.to_csv(f"{PAPER_DIR}/table3_recommendation_quality.csv",index=False)
print("Saved: table3_recommendation_quality.csv")

## 48. Sync all outputs to Drive & summary

In [ ]:
for fname in os.listdir(PAPER_DIR):
    shutil.copy2(f"{PAPER_DIR}/{fname}",f"{DRIVE_DIR}/paper/{fname}")
for fname in os.listdir("data/figures"):
    shutil.copy2(f"data/figures/{fname}",f"{DRIVE_DIR}/paper/{fname}")

print("All paper outputs synced to Drive.")

In [5]:
import json

with open("/content/drive/MyDrive/diabetes_project/Multimodal_System_ML_Research.ipynb", "r") as f:
    nb = json.load(f)

# Удалить проблемный ключ
if "widgets" in nb.get("metadata", {}):
    del nb["metadata"]["widgets"]
    print("Fixed: removed metadata.widgets")

with open("/content/Multimodal_System_ML_Research_fixed.ipynb", "w") as f:
    json.dump(nb, f, indent=1)

print("Done — download and upload to GitHub")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/diabetes_project/Multimodal_System_ML_Research.ipynb'